# Go1 IMU sensitivity — Colab runner

Runs the two GPU-bound steps of the IMU edge-case project:

1. **Train** a Go1 joystick policy (stock DR, and a no-noise variant).
2. **OFAT sweep** — inject idealised IMU errors one at a time and find where
   performance knees, then compare each knee against a real BMI270.

The sweep is the decision point. If no channel's knee is reachable by a real
device, the project's premise is false for this task and we learn it here
rather than in week 3.

**Set Runtime → Change runtime type → GPU before running anything.**

## 1. GPU check

Fail loudly rather than silently training on CPU for hours.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "NO GPU")

## 2. Install

The CUDA JAX wheel is the part people get wrong — the CPU wheel installs fine and runs 100x slower with no warning.

In [ ]:
# Do NOT install jax here. Colab already ships a working CUDA-enabled JAX, and# `pip install -U "jax[cuda12]"` upgrades jax/jaxlib while leaving the# preinstalled jax_cuda13_plugin at the old version. Both plugins then try to# register, cuda13 fails on an API mismatch, and cuda12 reports# ALREADY_EXISTS. It happens to still work, but it is a coin flip.## Install only what Colab lacks, and let pip resolve jax if a dependency# genuinely needs a different version.!pip install -q mujoco mujoco-mjx brax playground

In [ ]:
import jax, jaxlibprint("jax     ", jax.__version__)print("jaxlib  ", jaxlib.__version__)devs = jax.devices()print("devices:", devs)assert any(d.platform == "gpu" for d in devs), (    "No GPU visible to JAX.\n"    "  1. Runtime > Change runtime type > GPU, then Runtime > Restart session.\n"    "  2. If it still fails after installing brax (which can move jax), run:\n"    "       !pip uninstall -y jax-cuda13-plugin jax-cuda12-plugin\n"    "       !pip install -q -U 'jax[cuda12]'\n"    "     then Runtime > Restart session and re-run from the top.")assert jax.__version__ == jaxlib.__version__, (    f"jax {jax.__version__} != jaxlib {jaxlib.__version__}; mismatched versions "    "cause plugin registration errors. Reinstall both together.")print("OK -- JAX sees a GPU, versions match")

## 3. Mount Drive and cloneCheckpoints go to Drive. A Colab disconnect wipes the VM; a three-hourtraining run goes with it unless the weights are somewhere persistent.**The repo is private**, so Colab needs a token to clone it. Do NOT paste thetoken into a cell -- it would be saved in the notebook's output and history.Use Colab's secrets manager instead:1. Create a fine-grained GitHub PAT with **Contents: read** on this repo only   (github.com -> Settings -> Developer settings -> Personal access tokens).2. In Colab, click the key icon in the left sidebar.3. Add a secret named `GH_TOKEN`, paste the token, and enable notebook access.If you later make the repo public, the clone cell works unchanged with notoken at all.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
OUT = pathlib.Path('/content/drive/MyDrive/imu-diff-sim-runs')
OUT.mkdir(parents=True, exist_ok=True)
print("checkpoints ->", OUT)

In [ ]:
GH_USER = "stosifian"REPO    = "imu-diff-sim"# Token comes from Colab's secrets manager, never from a cell. Falls back to# an unauthenticated clone so this also works if the repo is made public.try:    from google.colab import userdata    _tok = userdata.get("GH_TOKEN")    url = f"https://{_tok}@github.com/{GH_USER}/{REPO}.git"    print("cloning with token from Colab secrets")except Exception as exc:    url = f"https://github.com/{GH_USER}/{REPO}.git"    print(f"no GH_TOKEN secret ({type(exc).__name__}); trying unauthenticated clone")import subprocess, pathlibdest = pathlib.Path("/content/imu-diff-sim")if not dest.exists():    r = subprocess.run(["git", "clone", "-q", url, str(dest)],                       capture_output=True, text=True)    # Never print the URL on failure: it contains the token.    assert r.returncode == 0, (        "clone failed. If the repo is private, check the GH_TOKEN secret "        "exists, has Contents:read on this repo, and notebook access is on.\n"        + r.stderr.replace(_tok, "***") if "_tok" in dir() else r.stderr    )print("cloned ->", dest)

In [ ]:
%cd /content/imu-diff-sim!git pull -q!pip install -q -e .

## 4. Sanity: the model still passes its tests on GPU

Cheap, and catches a bad install before an hour of training.

In [ ]:
!python -m pytest -q --no-header -x

## 5. Train

Two policies, because the comparison needs both:

- `stock` — the default `noise_config.level = 1.0`, i.e. uniform gyro DR of
  ±0.2 rad/s. This is standard practice, and it is 146× a real BMI270.
- `clean` — `level = 0.0`. Needed so the OFAT sweep measures injected errors
  against a clean floor rather than on top of noise 146× larger.

Start with a reduced `--timesteps` to check throughput before committing to a
full run. Watch the reward: Go1 joystick should climb steadily; if it is flat
after a few million steps something is wrong with the install.

In [ ]:
# Throughput check first — kill it once you can estimate steps/sec.
!python scripts/train_go1.py --timesteps 2000000 --noise-level 0.0 \
    --out /content/drive/MyDrive/imu-diff-sim-runs/go1_clean_smoke.pkl

In [ ]:
# Full runs. Expect tens of minutes each on a decent GPU.
!python scripts/train_go1.py --noise-level 0.0 \
    --out /content/drive/MyDrive/imu-diff-sim-runs/go1_clean.pkl

In [ ]:
!python scripts/train_go1.py --noise-level 1.0 \
    --out /content/drive/MyDrive/imu-diff-sim-runs/go1_stock.pkl

## 6. OFAT sweep — the actual question

Five factors, swept one at a time. The two that matter most are `gyro_white`
and `gyro_bias` at **equal magnitude**: white noise the policy can average
away over an episode (as 1/√N), a persistent bias it cannot. If bias degrades
performance at far lower magnitude, the structural argument is confirmed
empirically instead of asserted.

In [ ]:
!python scripts/ofat_sensitivity.py \
    --policy /content/drive/MyDrive/imu-diff-sim-runs/go1_clean.pkl \
    --episodes 64 --horizon 500 --points 9 \
    --out /content/drive/MyDrive/imu-diff-sim-runs/ofat_clean.npz

In [ ]:
# Same sweep against the stock-DR policy. Expect it to tolerate far more —
# that difference is the cost of the robustness margin.
!python scripts/ofat_sensitivity.py \
    --policy /content/drive/MyDrive/imu-diff-sim-runs/go1_stock.pkl \
    --episodes 64 --horizon 500 --points 9 \
    --out /content/drive/MyDrive/imu-diff-sim-runs/ofat_stock.npz

## 7. Read the result

The decision is the gap between each knee and the BMI270 line.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from imu_diff_sim.sim.go1_obs import (
    BMI270_GYRO_WHITE_STD_50HZ, PLAYGROUND_GYRO_DR_STD)

d = np.load('/content/drive/MyDrive/imu-diff-sim-runs/ofat_clean.npz')
factors = sorted({k.split('__')[0] for k in d.files})
fig, axes = plt.subplots(1, len(factors), figsize=(4*len(factors), 4),
                         constrained_layout=True)
for ax, f in zip(np.atleast_1d(axes), factors):
    m, r = d[f'{f}__magnitudes'], d[f'{f}__return']
    ax.plot(m, r / max(r.max(), 1e-9), 'o-')
    if f.startswith('gyro') and f != 'gyro_scale':
        ax.axvline(BMI270_GYRO_WHITE_STD_50HZ, color='green', ls='--',
                   label='BMI270')
        ax.axvline(PLAYGROUND_GYRO_DR_STD, color='crimson', ls=':',
                   label='Playground DR')
        ax.legend(fontsize=7)
    if f != 'latency':
        ax.set_xscale('log')
    ax.set_title(f); ax.set_xlabel('magnitude')
    ax.set_ylabel('normalised return'); ax.grid(alpha=.3)
plt.show()